# Step 1: Import Libraries

In [ ]:
"""
=============================================================
Task 02 - Customer Segmentation using K-Means Clustering
=============================================================
Dataset  : Mall_Customers.csv
Features : Annual Income (k$) & Spending Score (1-100)
Goal     : Group retail store customers based on purchase history
=============================================================
"""

# ─────────────────────────────────────────────
# STEP 1: Import Required Libraries
# ─────────────────────────────────────────────
# Yeh line notebook ke andar graphs ko auto-render karne ke liye hai
%matplotlib inline

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

print("=" * 60)
print("   TASK 02 — Customer Segmentation: K-Means Clustering")
print("=" * 60)


# ─────────────────────────────────────────────
# STEP 2: Load & Explore the Dataset
# ─────────────────────────────────────────────
print("\n[STEP 1] Loading Dataset...")

df = pd.read_csv("Mall_Customers.csv")

print(f"  ✔  Shape       : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  ✔  Columns     : {list(df.columns)}")
print(f"\n  First 5 rows:")
print(df.head().to_string(index=False))

print(f"\n  Dataset Info:")
print(f"  {'Column':<30} {'Non-Null':<12} {'Dtype'}")
print(f"  {'-'*50}")
for col in df.columns:
    print(f"  {col:<30} {df[col].notnull().sum():<12} {df[col].dtype}")

print(f"\n  Null Values : {df.isnull().sum().sum()}  (No missing data!)")
print(f"\n  Basic Statistics:")
print(df.describe().round(2).to_string())


# ─────────────────────────────────────────────
# STEP 3: Feature Selection
# ─────────────────────────────────────────────
print("\n[STEP 2] Selecting Features for Clustering...")

X = df[["Annual Income (k$)", "Spending Score (1-100)"]].values

print(f"  ✔  Selected Features : Annual Income (k$)  &  Spending Score (1-100)")
print(f"  ✔  Feature Matrix Shape : {X.shape}")


# ─────────────────────────────────────────────
# STEP 4: Feature Scaling
# ─────────────────────────────────────────────
print("\n[STEP 3] Standardizing Features (StandardScaler)...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"  ✔  Scaling Done  — Mean ≈ 0, Std ≈ 1 for each feature")


# ─────────────────────────────────────────────
# STEP 5: Elbow Method — Find Optimal K
# ─────────────────────────────────────────────
print("\n[STEP 4] Running Elbow Method to Find Optimal K...")

wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    print(f"  K={k:2d}  →  WCSS = {km.inertia_:.2f}")

print("\n  ✔  Optimal K identified at the 'elbow' of the curve.")


# ─────────────────────────────────────────────
# STEP 6: Silhouette Score Validation
# ─────────────────────────────────────────────
print("\n[STEP 5] Validating K using Silhouette Score...")

sil_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"  K={k:2d}  →  Silhouette Score = {score:.4f}")

best_k = np.argmax(sil_scores) + 2
print(f"\n  ✔  Best K by Silhouette Score = {best_k}")


# ─────────────────────────────────────────────
# STEP 7: Train Final K-Means Model (K=5)
# ─────────────────────────────────────────────
K = 5  # Optimal K from Elbow + Silhouette analysis

print(f"\n[STEP 6] Training Final K-Means Model with K = {K}...")

kmeans = KMeans(n_clusters=K, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

print(f"  ✔  Model Trained  — Inertia (WCSS) = {kmeans.inertia_:.2f}")
print(f"  ✔  Silhouette Score = {silhouette_score(X_scaled, df['Cluster']):.4f}")

print(f"\n  Cluster Distribution:")
cluster_counts = df["Cluster"].value_counts().sort_index()
for c, count in cluster_counts.items():
    print(f"     Cluster {c} : {count} customers")


# ─────────────────────────────────────────────
# STEP 8: Cluster Profiling
# ─────────────────────────────────────────────
print("\n[STEP 7] Profiling Each Cluster...")

cluster_names = {
    0: "Low Income – Low Spenders",
    1: "High Income – Low Spenders",
    2: "Average Income – Average Spenders",
    3: "Low Income – High Spenders",
    4: "High Income – High Spenders"
}

profile = df.groupby("Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(2)
profile["Count"] = cluster_counts
profile = profile.sort_index()

print(f"\n  {'Cluster':<10} {'Count':<8} {'Avg Age':<10} {'Avg Income':<15} {'Avg Spending'}")
print(f"  {'-'*60}")
for idx, row in profile.iterrows():
    print(f"  {idx:<10} {int(row['Count']):<8} {row['Age']:<10} {row['Annual Income (k$)']:<15} {row['Spending Score (1-100)']}")

# Assign labels based on income/spending pattern
income_avg = df.groupby("Cluster")["Annual Income (k$)"].mean()
spend_avg  = df.groupby("Cluster")["Spending Score (1-100)"].mean()
overall_income_avg = df["Annual Income (k$)"].mean()
overall_spend_avg  = df["Spending Score (1-100)"].mean()

def label_cluster(c):
    inc  = "High Income" if income_avg[c] >= overall_income_avg else "Low Income"
    spnd = "High Spenders" if spend_avg[c] >= overall_spend_avg else "Low Spenders"
    return f"{inc} – {spnd}"

print(f"\n  Cluster Labels (Auto-assigned):")
for c in sorted(df["Cluster"].unique()):
    print(f"     Cluster {c} : {label_cluster(c)}")


# ─────────────────────────────────────────────
# STEP 9: Visualizations (4 Plots in Notebook)
# ─────────────────────────────────────────────
print("\n[STEP 8] Generating Visualizations...")

COLORS = ["#FF6B6B", "#4ECDC4", "#FFE66D", "#A8DADC", "#F77F00"]
plt.rcParams.update({
    "figure.facecolor": "#0D1117",
    "axes.facecolor":   "#161B22",
    "axes.edgecolor":   "#30363D",
    "axes.labelcolor":  "#E6EDF3",
    "xtick.color":      "#8B949E",
    "ytick.color":      "#8B949E",
    "text.color":       "#E6EDF3",
    "grid.color":       "#21262D",
    "grid.alpha":       0.5,
})

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor("#0D1117")
fig.suptitle("Task 02 — Customer Segmentation using K-Means Clustering",
             fontsize=18, fontweight="bold", color="#58A6FF", y=0.98)

# ── Plot 1: Elbow Method ──────────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(list(K_range), wcss, "o-", color="#58A6FF", linewidth=2.5,
         markersize=7, markerfacecolor="#FF6B6B", markeredgecolor="white")
ax1.axvline(x=5, color="#FFE66D", linestyle="--", linewidth=1.5, alpha=0.8, label="Optimal K=5")
ax1.set_title("Elbow Method — Optimal K", fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("WCSS (Inertia)")
ax1.legend(facecolor="#21262D", edgecolor="#30363D")
ax1.grid(True)

# ── Plot 2: Silhouette Scores ─────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
k_vals = list(range(2, 11))
bars = ax2.bar(k_vals, sil_scores, color=[
    "#58A6FF" if k != best_k else "#FFE66D" for k in k_vals
], edgecolor="#30363D", linewidth=0.8)
ax2.set_title("Silhouette Score per K", fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_xticks(k_vals)
ax2.grid(True, axis="y")

# ── Plot 3: Main Cluster Scatter ──────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
for c in range(K):
    mask = df["Cluster"] == c
    ax3.scatter(df.loc[mask, "Annual Income (k$)"],
                df.loc[mask, "Spending Score (1-100)"],
                c=COLORS[c], s=80, alpha=0.85, edgecolors="white",
                linewidths=0.4, label=f"Cluster {c}", zorder=3)

# Plot centroids (inverse transform back to original scale)
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
ax3.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
            c="white", s=250, marker="*", edgecolors="#FFE66D",
            linewidths=1.2, zorder=5, label="Centroids")

ax3.set_title("Customer Clusters\n(Annual Income vs Spending Score)",
              fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax3.set_xlabel("Annual Income (k$)")
ax3.set_ylabel("Spending Score (1-100)")
ax3.legend(facecolor="#21262D", edgecolor="#30363D", fontsize=9)
ax3.grid(True)

# ── Plot 4: Cluster Size Bar Chart ───────────────────────
ax4 = fig.add_subplot(2, 2, 4)
sizes = [cluster_counts[c] for c in range(K)]
labels_short = [f"C{c}" for c in range(K)]
ax4.bar(labels_short, sizes, color=COLORS, edgecolor="#30363D", linewidth=0.8)
for i, (l, s) in enumerate(zip(labels_short, sizes)):
    ax4.text(i, s + 0.5, str(s), ha="center", va="bottom",
             fontsize=11, fontweight="bold", color="white")
ax4.set_title("Number of Customers per Cluster",
              fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax4.set_xlabel("Cluster")
ax4.set_ylabel("Customer Count")
ax4.grid(True, axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.96])

# Plot ko save bhi karega backend par
plt.savefig("kmeans_results.png", dpi=150, bbox_inches="tight", facecolor="#0D1117")

# BADLAV: plt.close() ko hata kar plt.show() lagaya taaki graph notebook me dikhe
plt.show() 
print("  ✔  Plot saved as  →  kmeans_results.png and displayed above!")


# ─────────────────────────────────────────────
# STEP 10: Save Results to CSV
# ─────────────────────────────────────────────
print("\n[STEP 9] Saving Clustered Data...")

output = df.copy()
output["Cluster_Label"] = output["Cluster"].map(lambda c: label_cluster(c))
output.to_csv("Mall_Customers_Clustered.csv", index=False)
print("  ✔  Results saved as  →  Mall_Customers_Clustered.csv")


# ─────────────────────────────────────────────
# STEP 11: Final Summary
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("    FINAL SUMMARY")
print("=" * 60)
print(f"  Algorithm        : K-Means Clustering (k-means++ init)")
print(f"  Optimal K        : {K}")
print(f"  Total Customers : {len(df)}")
print(f"  WCSS (Inertia)  : {kmeans.inertia_:.4f}")
print(f"  Silhouette Score: {silhouette_score(X_scaled, df['Cluster']):.4f}")
print(f"\n  Cluster Breakdown:")
for c in range(K):
    cnt = cluster_counts[c]
    pct = (cnt / len(df)) * 100
    print(f"     Cluster {c} ({cnt} customers, {pct:.1f}%) → {label_cluster(c)}")

print("\n  Output Files:")
print("     ✔  kmeans_results.png          (4-panel visualization)")
print("     ✔  Mall_Customers_Clustered.csv (data with cluster labels)")
print("\n" + "=" * 60)
print("    Task 02 Completed Successfully!")
print("=" * 60)

In [2]:
"""
=============================================================
Task 02 — Customer Segmentation using K-Means Clustering
ALL POSSIBLE GRAPHS (14 Visualizations)
Dataset: Mall_Customers.csv
=============================================================
Required Libraries:
    pip install pandas scikit-learn matplotlib seaborn numpy
=============================================================
"""

# ─────────────────────────────────────────────────────────────
# STEP 1: Import Libraries
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# STEP 2: Global Style Settings
# ─────────────────────────────────────────────────────────────
DARK_BG    = "#0D1117"
CARD_BG    = "#161B22"
BORDER     = "#30363D"
TEXT_MAIN  = "#E6EDF3"
TEXT_SUB   = "#8B949E"
BLUE       = "#58A6FF"
YELLOW     = "#FFE66D"

CLUSTER_COLORS = ["#FF6B6B", "#4ECDC4", "#FFE66D", "#A8DADC", "#F77F00"]

plt.rcParams.update({
    "figure.facecolor":  DARK_BG,
    "axes.facecolor":    CARD_BG,
    "axes.edgecolor":    BORDER,
    "axes.labelcolor":   TEXT_MAIN,
    "axes.titlecolor":   BLUE,
    "axes.titlesize":    13,
    "axes.titleweight":  "bold",
    "axes.titlepad":     14,
    "xtick.color":       TEXT_SUB,
    "ytick.color":       TEXT_SUB,
    "text.color":        TEXT_MAIN,
    "grid.color":        "#21262D",
    "grid.alpha":        0.6,
    "legend.facecolor":  "#21262D",
    "legend.edgecolor":  BORDER,
    "legend.fontsize":   9,
    "font.family":       "monospace",
})

def title_fig(fig, title):
    fig.suptitle(title, fontsize=16, fontweight="bold",
                 color=BLUE, y=0.98)

def save(fig, filename):
    fig.savefig(filename, dpi=140, bbox_inches="tight", facecolor=DARK_BG)
    plt.close(fig)
    print(f"  ✔  Saved  →  {filename}")

# ─────────────────────────────────────────────────────────────
# STEP 3: Load Dataset
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("   TASK 02 — K-Means Clustering  |  ALL GRAPHS")
print("=" * 65)
print("\n[STEP 1] Loading Dataset...")

df = pd.read_csv("Mall_Customers.csv")

print(f"  Shape   : {df.shape}")
print(f"  Columns : {list(df.columns)}")
print(f"  Nulls   : {df.isnull().sum().sum()}")
print(df.head(3).to_string(index=False))

# ─────────────────────────────────────────────────────────────
# STEP 4: Feature Selection & Scaling
# ─────────────────────────────────────────────────────────────
print("\n[STEP 2] Feature Selection & Scaling...")

X = df[["Annual Income (k$)", "Spending Score (1-100)"]].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("  ✔  Features scaled with StandardScaler")

# ─────────────────────────────────────────────────────────────
# STEP 5: Find Optimal K
# ─────────────────────────────────────────────────────────────
print("\n[STEP 3] Finding Optimal K (Elbow + Silhouette)...")

K_range  = range(1, 11)
wcss     = []
sil_list = []

for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    if k >= 2:
        sil_list.append(silhouette_score(X_scaled, km.labels_))

K = 5  # Optimal (confirmed by Elbow + Silhouette)
print(f"  ✔  Optimal K = {K}")

# ─────────────────────────────────────────────────────────────
# STEP 6: Train Final Model
# ─────────────────────────────────────────────────────────────
print("\n[STEP 4] Training Final K-Means Model (K=5)...")

kmeans = KMeans(n_clusters=K, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)
centroids_scaled = kmeans.cluster_centers_
centroids_orig   = scaler.inverse_transform(centroids_scaled)

final_sil = silhouette_score(X_scaled, df["Cluster"])
print(f"  ✔  Inertia (WCSS)    = {kmeans.inertia_:.2f}")
print(f"  ✔  Silhouette Score  = {final_sil:.4f}")

cluster_labels = {
    c: f"Cluster {c}" for c in range(K)
}
df["Cluster_Name"] = df["Cluster"].map(cluster_labels)

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 1 — Elbow Method (WCSS)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
print("\n[GRAPHS] Generating all visualizations...")

fig, ax = plt.subplots(figsize=(10, 6))
title_fig(fig, "Graph 1 — Elbow Method: Finding Optimal K")

ax.plot(list(K_range), wcss, "o-", color=BLUE, linewidth=2.5,
        markersize=9, markerfacecolor="#FF6B6B",
        markeredgecolor="white", markeredgewidth=1.2)
ax.axvline(x=5, color=YELLOW, linestyle="--",
           linewidth=2, alpha=0.85, label="Optimal K = 5")
ax.fill_between(list(K_range), wcss, alpha=0.08, color=BLUE)

for i, (k, w) in enumerate(zip(K_range, wcss)):
    ax.annotate(f"{w:.1f}", (k, w),
                textcoords="offset points", xytext=(0, 12),
                ha="center", fontsize=8, color=TEXT_SUB)

ax.set_xlabel("Number of Clusters (K)", fontsize=11)
ax.set_ylabel("WCSS — Within Cluster Sum of Squares", fontsize=11)
ax.set_xticks(list(K_range))
ax.legend()
ax.grid(True)
save(fig, "graph_01_elbow_method.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 2 — Silhouette Score per K
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
title_fig(fig, "Graph 2 — Silhouette Score per K")

k_vals = list(range(2, 11))
bar_colors = [YELLOW if k == 5 else BLUE for k in k_vals]
bars = ax.bar(k_vals, sil_list, color=bar_colors,
              edgecolor=BORDER, linewidth=0.8, width=0.6)

for bar, score in zip(bars, sil_list):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.005,
            f"{score:.3f}", ha="center", va="bottom",
            fontsize=9, color=TEXT_MAIN, fontweight="bold")

ax.set_xlabel("Number of Clusters (K)", fontsize=11)
ax.set_ylabel("Silhouette Score", fontsize=11)
ax.set_xticks(k_vals)
ax.axhline(y=max(sil_list), color=YELLOW,
           linestyle="--", linewidth=1.2, alpha=0.7,
           label=f"Best Score = {max(sil_list):.3f} at K=5")
ax.legend()
ax.grid(True, axis="y")
save(fig, "graph_02_silhouette_score.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 3 — Main Cluster Scatter Plot
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
title_fig(fig, "Graph 3 — Customer Clusters\n(Annual Income vs Spending Score)")

for c in range(K):
    mask = df["Cluster"] == c
    ax.scatter(df.loc[mask, "Annual Income (k$)"],
               df.loc[mask, "Spending Score (1-100)"],
               c=CLUSTER_COLORS[c], s=90, alpha=0.88,
               edgecolors="white", linewidths=0.4,
               label=f"Cluster {c}  (n={mask.sum()})", zorder=3)

ax.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
           c="white", s=300, marker="*",
           edgecolors=YELLOW, linewidths=1.5,
           zorder=5, label="Centroids")

ax.set_xlabel("Annual Income (k$)", fontsize=11)
ax.set_ylabel("Spending Score (1-100)", fontsize=11)
ax.legend()
ax.grid(True)
save(fig, "graph_03_cluster_scatter.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 4 — Cluster Size (Bar Chart)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
title_fig(fig, "Graph 4 — Number of Customers per Cluster")

counts = df["Cluster"].value_counts().sort_index()
bars = ax.bar([f"Cluster {c}" for c in counts.index],
              counts.values, color=CLUSTER_COLORS,
              edgecolor=BORDER, linewidth=0.8)

for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.6,
            f"{val}\n({val/len(df)*100:.1f}%)",
            ha="center", va="bottom",
            fontsize=10, fontweight="bold", color=TEXT_MAIN)

ax.set_ylabel("Number of Customers", fontsize=11)
ax.set_xlabel("Cluster", fontsize=11)
ax.grid(True, axis="y")
save(fig, "graph_04_cluster_sizes.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 5 — Cluster Size (Pie Chart)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 9))
title_fig(fig, "Graph 5 — Customer Distribution per Cluster (Pie Chart)")

explode = [0.04] * K
wedges, texts, autotexts = ax.pie(
    counts.values,
    labels=[f"Cluster {c}" for c in counts.index],
    colors=CLUSTER_COLORS,
    autopct="%1.1f%%",
    startangle=140,
    explode=explode,
    wedgeprops=dict(edgecolor=DARK_BG, linewidth=2)
)
for t in texts:
    t.set_color(TEXT_MAIN)
    t.set_fontsize(11)
for at in autotexts:
    at.set_color(DARK_BG)
    at.set_fontsize(10)
    at.set_fontweight("bold")

save(fig, "graph_05_pie_chart.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 6 — Age Distribution per Cluster (Box Plot)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
title_fig(fig, "Graph 6 — Age Distribution per Cluster (Box Plot)")

data_by_cluster = [df[df["Cluster"] == c]["Age"].values for c in range(K)]
bp = ax.boxplot(data_by_cluster,
                patch_artist=True,
                medianprops=dict(color=YELLOW, linewidth=2.5),
                whiskerprops=dict(color=TEXT_SUB),
                capprops=dict(color=TEXT_SUB),
                flierprops=dict(markerfacecolor=TEXT_SUB, markersize=5))

for patch, color in zip(bp["boxes"], CLUSTER_COLORS):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticklabels([f"Cluster {c}" for c in range(K)])
ax.set_ylabel("Age", fontsize=11)
ax.set_xlabel("Cluster", fontsize=11)
ax.grid(True, axis="y")
save(fig, "graph_06_age_boxplot.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 7 — Annual Income Distribution per Cluster (Violin)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
title_fig(fig, "Graph 7 — Annual Income Distribution per Cluster (Violin Plot)")

vp = ax.violinplot(
    [df[df["Cluster"] == c]["Annual Income (k$)"].values for c in range(K)],
    positions=range(K),
    showmeans=True,
    showmedians=True
)
for i, (body, color) in enumerate(zip(vp["bodies"], CLUSTER_COLORS)):
    body.set_facecolor(color)
    body.set_alpha(0.65)

vp["cmeans"].set_color(YELLOW)
vp["cmedians"].set_color("white")
vp["cbars"].set_color(BORDER)
vp["cmins"].set_color(BORDER)
vp["cmaxes"].set_color(BORDER)

ax.set_xticks(range(K))
ax.set_xticklabels([f"Cluster {c}" for c in range(K)])
ax.set_ylabel("Annual Income (k$)", fontsize=11)
ax.set_xlabel("Cluster", fontsize=11)
ax.grid(True, axis="y")
mean_patch = mpatches.Patch(color=YELLOW, label="Mean")
med_patch  = mpatches.Patch(color="white", label="Median")
ax.legend(handles=[mean_patch, med_patch])
save(fig, "graph_07_income_violin.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 8 — Spending Score Distribution (Violin)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
title_fig(fig, "Graph 8 — Spending Score Distribution per Cluster (Violin Plot)")

vp2 = ax.violinplot(
    [df[df["Cluster"] == c]["Spending Score (1-100)"].values for c in range(K)],
    positions=range(K),
    showmeans=True,
    showmedians=True
)
for body, color in zip(vp2["bodies"], CLUSTER_COLORS):
    body.set_facecolor(color)
    body.set_alpha(0.65)

vp2["cmeans"].set_color(YELLOW)
vp2["cmedians"].set_color("white")
vp2["cbars"].set_color(BORDER)
vp2["cmins"].set_color(BORDER)
vp2["cmaxes"].set_color(BORDER)

ax.set_xticks(range(K))
ax.set_xticklabels([f"Cluster {c}" for c in range(K)])
ax.set_ylabel("Spending Score (1-100)", fontsize=11)
ax.set_xlabel("Cluster", fontsize=11)
ax.grid(True, axis="y")
ax.legend(handles=[mean_patch, med_patch])
save(fig, "graph_08_spending_violin.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 9 — Gender Distribution per Cluster (Stacked Bar)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
title_fig(fig, "Graph 9 — Gender Distribution per Cluster (Stacked Bar)")

gender_cluster = df.groupby(["Cluster", "Gender"]).size().unstack(fill_value=0)
x = np.arange(K)
male_vals   = gender_cluster.get("Male",   pd.Series([0]*K)).values
female_vals = gender_cluster.get("Female", pd.Series([0]*K)).values

bars_m = ax.bar(x, male_vals,   color="#58A6FF", label="Male",
                edgecolor=BORDER, linewidth=0.8)
bars_f = ax.bar(x, female_vals, bottom=male_vals,
                color="#FF6B6B", label="Female",
                edgecolor=BORDER, linewidth=0.8)

for bar, val in zip(bars_m, male_vals):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width()/2, val/2,
                str(val), ha="center", va="center",
                fontsize=10, fontweight="bold", color=DARK_BG)

for bar, mval, fval in zip(bars_f, male_vals, female_vals):
    if fval > 0:
        ax.text(bar.get_x() + bar.get_width()/2, mval + fval/2,
                str(fval), ha="center", va="center",
                fontsize=10, fontweight="bold", color=DARK_BG)

ax.set_xticks(x)
ax.set_xticklabels([f"Cluster {c}" for c in range(K)])
ax.set_ylabel("Number of Customers", fontsize=11)
ax.set_xlabel("Cluster", fontsize=11)
ax.legend()
ax.grid(True, axis="y")
save(fig, "graph_09_gender_distribution.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 10 — Cluster Mean Radar / Bar Profile
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
title_fig(fig, "Graph 10 — Cluster Profile: Avg Age, Income & Spending Score")

profile = df.groupby("Cluster")[
    ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
].mean().round(1)

metrics  = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
x_pos    = np.arange(len(metrics))
width    = 0.15
offsets  = np.linspace(-(K-1)*width/2, (K-1)*width/2, K)

for c in range(K):
    vals = [profile.loc[c, m] for m in metrics]
    ax.bar(x_pos + offsets[c], vals,
           width=width, color=CLUSTER_COLORS[c],
           edgecolor=BORDER, linewidth=0.6,
           label=f"Cluster {c}")

ax.set_xticks(x_pos)
ax.set_xticklabels(metrics, fontsize=10)
ax.set_ylabel("Average Value", fontsize=11)
ax.legend(loc="upper right")
ax.grid(True, axis="y")
save(fig, "graph_10_cluster_profile_bar.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 11 — Correlation Heatmap
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
title_fig(fig, "Graph 11 — Feature Correlation Heatmap")

corr = df[["Age", "Annual Income (k$)",
           "Spending Score (1-100)", "Cluster"]].corr()

cmap = LinearSegmentedColormap.from_list(
    "custom", [CLUSTER_COLORS[0], CARD_BG, BLUE])
sns.heatmap(corr, annot=True, fmt=".2f", cmap=cmap,
            ax=ax, linewidths=0.5, linecolor=BORDER,
            annot_kws={"size": 11, "weight": "bold"},
            cbar_kws={"shrink": 0.8})
ax.set_title("Graph 11 — Feature Correlation Heatmap",
             fontsize=13, fontweight="bold", color=BLUE, pad=14)
save(fig, "graph_11_heatmap.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 12 — Silhouette Plot (per-sample)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
title_fig(fig, "Graph 12 — Silhouette Plot (Per-Sample Analysis)")

sil_vals = silhouette_samples(X_scaled, df["Cluster"])
y_lower  = 10

for c in range(K):
    c_sil = np.sort(sil_vals[df["Cluster"] == c])
    size  = c_sil.shape[0]
    y_upper = y_lower + size

    ax.fill_betweenx(np.arange(y_lower, y_upper),
                     0, c_sil, facecolor=CLUSTER_COLORS[c], alpha=0.8)
    ax.text(-0.05, y_lower + size / 2,
            f"C{c}", color=TEXT_MAIN, fontsize=10)
    y_lower = y_upper + 10

ax.axvline(x=final_sil, color=YELLOW, linestyle="--",
           linewidth=2, label=f"Avg Silhouette = {final_sil:.3f}")
ax.set_xlabel("Silhouette Coefficient", fontsize=11)
ax.set_ylabel("Cluster Samples", fontsize=11)
ax.set_yticks([])
ax.legend()
ax.grid(True, axis="x")
save(fig, "graph_12_silhouette_plot.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 13 — PCA 2D Cluster View
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
title_fig(fig, "Graph 13 — PCA 2D Projection of Clusters")

X_all = df[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].values
X_all_scaled = StandardScaler().fit_transform(X_all)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_all_scaled)

for c in range(K):
    mask = df["Cluster"] == c
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=CLUSTER_COLORS[c], s=80, alpha=0.85,
               edgecolors="white", linewidths=0.4,
               label=f"Cluster {c}  (n={mask.sum()})", zorder=3)

explained = pca.explained_variance_ratio_
ax.set_xlabel(f"Principal Component 1  ({explained[0]*100:.1f}% variance)", fontsize=10)
ax.set_ylabel(f"Principal Component 2  ({explained[1]*100:.1f}% variance)", fontsize=10)
ax.legend()
ax.grid(True)
save(fig, "graph_13_pca_projection.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 14 — Pair Plot (Income, Spending, Age colored by Cluster)
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
print("  Generating pair plot (may take a moment)...")

palette = {c: CLUSTER_COLORS[c] for c in range(K)}
pair_df = df[["Age", "Annual Income (k$)",
              "Spending Score (1-100)", "Cluster"]].copy()

with plt.rc_context({
    "figure.facecolor": DARK_BG,
    "axes.facecolor":   CARD_BG,
    "axes.edgecolor":   BORDER,
    "axes.labelcolor":  TEXT_MAIN,
    "xtick.color":      TEXT_SUB,
    "ytick.color":      TEXT_SUB,
    "text.color":       TEXT_MAIN,
    "grid.color":       "#21262D",
    "grid.alpha":       0.6,
}):
    g = sns.pairplot(pair_df, hue="Cluster",
                     palette=palette,
                     plot_kws=dict(alpha=0.7, edgecolor="none", s=40),
                     diag_kws=dict(alpha=0.6))
    g.figure.patch.set_facecolor(DARK_BG)
    for ax_pp in g.axes.flatten():
        if ax_pp:
            ax_pp.set_facecolor(CARD_BG)

    g.figure.suptitle(
        "Graph 14 — Pair Plot: Age, Annual Income & Spending Score by Cluster",
        y=1.02, fontsize=14, fontweight="bold", color=BLUE)

    g.figure.savefig("graph_14_pair_plot.png", dpi=130,
                     bbox_inches="tight", facecolor=DARK_BG)
    plt.close(g.figure)
    print("  ✔  Saved  →  graph_14_pair_plot.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 15 — Age vs Spending Score Scatter by Cluster
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
title_fig(fig, "Graph 15 — Age vs Spending Score by Cluster")

for c in range(K):
    mask = df["Cluster"] == c
    ax.scatter(df.loc[mask, "Age"],
               df.loc[mask, "Spending Score (1-100)"],
               c=CLUSTER_COLORS[c], s=90, alpha=0.85,
               edgecolors="white", linewidths=0.4,
               label=f"Cluster {c}  (n={mask.sum()})")

ax.set_xlabel("Age", fontsize=11)
ax.set_ylabel("Spending Score (1-100)", fontsize=11)
ax.legend()
ax.grid(True)
save(fig, "graph_15_age_vs_spending.png")

# ─────────────────────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════
#  GRAPH 16 — Age vs Annual Income Scatter by Cluster
# ══════════════════════════════════════════════════════════════
# ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
title_fig(fig, "Graph 16 — Age vs Annual Income by Cluster")

for c in range(K):
    mask = df["Cluster"] == c
    ax.scatter(df.loc[mask, "Age"],
               df.loc[mask, "Annual Income (k$)"],
               c=CLUSTER_COLORS[c], s=90, alpha=0.85,
               edgecolors="white", linewidths=0.4,
               label=f"Cluster {c}  (n={mask.sum()})")

ax.set_xlabel("Age", fontsize=11)
ax.set_ylabel("Annual Income (k$)", fontsize=11)
ax.legend()
ax.grid(True)
save(fig, "graph_16_age_vs_income.png")

# ─────────────────────────────────────────────────────────────
# FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("   FINAL SUMMARY")
print("=" * 65)
print(f"  Algorithm       : K-Means  (k-means++ initialization)")
print(f"  Optimal K       : {K}")
print(f"  Total Customers : {len(df)}")
print(f"  WCSS (Inertia)  : {kmeans.inertia_:.4f}")
print(f"  Silhouette Score: {final_sil:.4f}")

print(f"\n  Cluster Breakdown:")
profile = df.groupby("Cluster")[
    ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
].mean().round(1)
counts_final = df["Cluster"].value_counts().sort_index()
for c in range(K):
    cnt = counts_final[c]
    pct = cnt / len(df) * 100
    avg_inc  = profile.loc[c, "Annual Income (k$)"]
    avg_spnd = profile.loc[c, "Spending Score (1-100)"]
    print(f"    Cluster {c}  |  {cnt:3d} customers ({pct:.1f}%)  "
          f"|  Avg Income: {avg_inc}k$  |  Avg Spending: {avg_spnd}")

print(f"\n  16 Graphs Saved:")
graphs = [
    "graph_01_elbow_method.png          — Elbow Method (WCSS)",
    "graph_02_silhouette_score.png       — Silhouette Score per K",
    "graph_03_cluster_scatter.png        — Main Cluster Scatter",
    "graph_04_cluster_sizes.png          — Cluster Size Bar Chart",
    "graph_05_pie_chart.png              — Cluster Pie Chart",
    "graph_06_age_boxplot.png            — Age Box Plot per Cluster",
    "graph_07_income_violin.png          — Income Violin Plot",
    "graph_08_spending_violin.png        — Spending Violin Plot",
    "graph_09_gender_distribution.png    — Gender Stacked Bar",
    "graph_10_cluster_profile_bar.png    — Cluster Profile Bar",
    "graph_11_heatmap.png                — Correlation Heatmap",
    "graph_12_silhouette_plot.png        — Per-Sample Silhouette",
    "graph_13_pca_projection.png         — PCA 2D Projection",
    "graph_14_pair_plot.png              — Pair Plot (all features)",
    "graph_15_age_vs_spending.png        — Age vs Spending Score",
    "graph_16_age_vs_income.png          — Age vs Annual Income",
]
for g in graphs:
    print(f"    ✔  {g}")

print("\n" + "=" * 65)
print("   ALL DONE — Task 02 Completed Successfully!")
print("=" * 65)

   TASK 02 — K-Means Clustering  |  ALL GRAPHS

[STEP 1] Loading Dataset...
  Shape   : (200, 5)
  Columns : ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
  Nulls   : 0
 CustomerID Gender  Age  Annual Income (k$)  Spending Score (1-100)
          1   Male   19                  15                      39
          2   Male   21                  15                      81
          3 Female   20                  16                       6

[STEP 2] Feature Selection & Scaling...
  ✔  Features scaled with StandardScaler

[STEP 3] Finding Optimal K (Elbow + Silhouette)...
  ✔  Optimal K = 5

[STEP 4] Training Final K-Means Model (K=5)...
  ✔  Inertia (WCSS)    = 65.57
  ✔  Silhouette Score  = 0.5547

[GRAPHS] Generating all visualizations...
  ✔  Saved  →  graph_01_elbow_method.png
  ✔  Saved  →  graph_02_silhouette_score.png
  ✔  Saved  →  graph_03_cluster_scatter.png
  ✔  Saved  →  graph_04_cluster_sizes.png
  ✔  Saved  →  graph_05_pie_chart.png
  ✔  Sav

In [1]:
"""
=============================================================
Task 02 - Customer Segmentation using K-Means Clustering
=============================================================
Dataset  : Mall_Customers.csv
Features : Annual Income (k$) & Spending Score (1-100)
Goal     : Group retail store customers based on purchase history
=============================================================
"""

# ─────────────────────────────────────────────
# STEP 1: Import Required Libraries
# ─────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

print("=" * 60)
print("   TASK 02 — Customer Segmentation: K-Means Clustering")
print("=" * 60)


# ─────────────────────────────────────────────
# STEP 2: Load & Explore the Dataset
# ─────────────────────────────────────────────
print("\n[STEP 1] Loading Dataset...")

df = pd.read_csv("Mall_Customers.csv")

print(f"  ✔  Shape       : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"  ✔  Columns     : {list(df.columns)}")
print(f"\n  First 5 rows:")
print(df.head().to_string(index=False))

print(f"\n  Dataset Info:")
print(f"  {'Column':<30} {'Non-Null':<12} {'Dtype'}")
print(f"  {'-'*50}")
for col in df.columns:
    print(f"  {col:<30} {df[col].notnull().sum():<12} {df[col].dtype}")

print(f"\n  Null Values : {df.isnull().sum().sum()}  (No missing data!)")
print(f"\n  Basic Statistics:")
print(df.describe().round(2).to_string())


# ─────────────────────────────────────────────
# STEP 3: Feature Selection
# ─────────────────────────────────────────────
print("\n[STEP 2] Selecting Features for Clustering...")

X = df[["Annual Income (k$)", "Spending Score (1-100)"]].values

print(f"  ✔  Selected Features : Annual Income (k$)  &  Spending Score (1-100)")
print(f"  ✔  Feature Matrix Shape : {X.shape}")


# ─────────────────────────────────────────────
# STEP 4: Feature Scaling
# ─────────────────────────────────────────────
print("\n[STEP 3] Standardizing Features (StandardScaler)...")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"  ✔  Scaling Done  — Mean ≈ 0, Std ≈ 1 for each feature")


# ─────────────────────────────────────────────
# STEP 5: Elbow Method — Find Optimal K
# ─────────────────────────────────────────────
print("\n[STEP 4] Running Elbow Method to Find Optimal K...")

wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
    km.fit(X_scaled)
    wcss.append(km.inertia_)
    print(f"  K={k:2d}  →  WCSS = {km.inertia_:.2f}")

print("\n  ✔  Optimal K identified at the 'elbow' of the curve.")


# ─────────────────────────────────────────────
# STEP 6: Silhouette Score Validation
# ─────────────────────────────────────────────
print("\n[STEP 5] Validating K using Silhouette Score...")

sil_scores = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    sil_scores.append(score)
    print(f"  K={k:2d}  →  Silhouette Score = {score:.4f}")

best_k = np.argmax(sil_scores) + 2
print(f"\n  ✔  Best K by Silhouette Score = {best_k}")


# ─────────────────────────────────────────────
# STEP 7: Train Final K-Means Model (K=5)
# ─────────────────────────────────────────────
K = 5  # Optimal K from Elbow + Silhouette analysis

print(f"\n[STEP 6] Training Final K-Means Model with K = {K}...")

kmeans = KMeans(n_clusters=K, init="k-means++", max_iter=300,
                n_init=10, random_state=42)
df["Cluster"] = kmeans.fit_predict(X_scaled)

print(f"  ✔  Model Trained  — Inertia (WCSS) = {kmeans.inertia_:.2f}")
print(f"  ✔  Silhouette Score = {silhouette_score(X_scaled, df['Cluster']):.4f}")

print(f"\n  Cluster Distribution:")
cluster_counts = df["Cluster"].value_counts().sort_index()
for c, count in cluster_counts.items():
    print(f"    Cluster {c} : {count} customers")


# ─────────────────────────────────────────────
# STEP 8: Cluster Profiling
# ─────────────────────────────────────────────
print("\n[STEP 7] Profiling Each Cluster...")

cluster_names = {
    0: "Low Income – Low Spenders",
    1: "High Income – Low Spenders",
    2: "Average Income – Average Spenders",
    3: "Low Income – High Spenders",
    4: "High Income – High Spenders"
}

profile = df.groupby("Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(2)
profile["Count"] = cluster_counts
profile = profile.sort_index()

print(f"\n  {'Cluster':<10} {'Count':<8} {'Avg Age':<10} {'Avg Income':<15} {'Avg Spending'}")
print(f"  {'-'*60}")
for idx, row in profile.iterrows():
    print(f"  {idx:<10} {int(row['Count']):<8} {row['Age']:<10} {row['Annual Income (k$)']:<15} {row['Spending Score (1-100)']}")

# Assign labels based on income/spending pattern
income_avg = df.groupby("Cluster")["Annual Income (k$)"].mean()
spend_avg  = df.groupby("Cluster")["Spending Score (1-100)"].mean()
overall_income_avg = df["Annual Income (k$)"].mean()
overall_spend_avg  = df["Spending Score (1-100)"].mean()

def label_cluster(c):
    inc  = "High Income" if income_avg[c] >= overall_income_avg else "Low Income"
    spnd = "High Spenders" if spend_avg[c] >= overall_spend_avg else "Low Spenders"
    return f"{inc} – {spnd}"

print(f"\n  Cluster Labels (Auto-assigned):")
for c in sorted(df["Cluster"].unique()):
    print(f"    Cluster {c} : {label_cluster(c)}")


# ─────────────────────────────────────────────
# STEP 9: Visualizations (4 Plots)
# ─────────────────────────────────────────────
print("\n[STEP 8] Generating Visualizations...")

COLORS = ["#FF6B6B", "#4ECDC4", "#FFE66D", "#A8DADC", "#F77F00"]
plt.rcParams.update({
    "figure.facecolor": "#0D1117",
    "axes.facecolor":   "#161B22",
    "axes.edgecolor":   "#30363D",
    "axes.labelcolor":  "#E6EDF3",
    "xtick.color":      "#8B949E",
    "ytick.color":      "#8B949E",
    "text.color":       "#E6EDF3",
    "grid.color":       "#21262D",
    "grid.alpha":       0.5,
})

fig = plt.figure(figsize=(18, 14))
fig.patch.set_facecolor("#0D1117")
fig.suptitle("Task 02 — Customer Segmentation using K-Means Clustering",
             fontsize=18, fontweight="bold", color="#58A6FF", y=0.98)

# ── Plot 1: Elbow Method ──────────────────────────────────
ax1 = fig.add_subplot(2, 2, 1)
ax1.plot(list(K_range), wcss, "o-", color="#58A6FF", linewidth=2.5,
         markersize=7, markerfacecolor="#FF6B6B", markeredgecolor="white")
ax1.axvline(x=5, color="#FFE66D", linestyle="--", linewidth=1.5, alpha=0.8, label="Optimal K=5")
ax1.set_title("Elbow Method — Optimal K", fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax1.set_xlabel("Number of Clusters (K)")
ax1.set_ylabel("WCSS (Inertia)")
ax1.legend(facecolor="#21262D", edgecolor="#30363D")
ax1.grid(True)

# ── Plot 2: Silhouette Scores ─────────────────────────────
ax2 = fig.add_subplot(2, 2, 2)
k_vals = list(range(2, 11))
bars = ax2.bar(k_vals, sil_scores, color=[
    "#58A6FF" if k != best_k else "#FFE66D" for k in k_vals
], edgecolor="#30363D", linewidth=0.8)
ax2.set_title("Silhouette Score per K", fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax2.set_xlabel("Number of Clusters (K)")
ax2.set_ylabel("Silhouette Score")
ax2.set_xticks(k_vals)
ax2.grid(True, axis="y")

# ── Plot 3: Main Cluster Scatter ──────────────────────────
ax3 = fig.add_subplot(2, 2, 3)
for c in range(K):
    mask = df["Cluster"] == c
    ax3.scatter(df.loc[mask, "Annual Income (k$)"],
                df.loc[mask, "Spending Score (1-100)"],
                c=COLORS[c], s=80, alpha=0.85, edgecolors="white",
                linewidths=0.4, label=f"Cluster {c}", zorder=3)

# Plot centroids (inverse transform back to original scale)
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
ax3.scatter(centroids_orig[:, 0], centroids_orig[:, 1],
            c="white", s=250, marker="*", edgecolors="#FFE66D",
            linewidths=1.2, zorder=5, label="Centroids")

ax3.set_title("Customer Clusters\n(Annual Income vs Spending Score)",
              fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax3.set_xlabel("Annual Income (k$)")
ax3.set_ylabel("Spending Score (1-100)")
ax3.legend(facecolor="#21262D", edgecolor="#30363D", fontsize=9)
ax3.grid(True)

# ── Plot 4: Cluster Size Bar Chart ───────────────────────
ax4 = fig.add_subplot(2, 2, 4)
sizes = [cluster_counts[c] for c in range(K)]
labels_short = [f"C{c}" for c in range(K)]
ax4.bar(labels_short, sizes, color=COLORS, edgecolor="#30363D", linewidth=0.8)
for i, (l, s) in enumerate(zip(labels_short, sizes)):
    ax4.text(i, s + 0.5, str(s), ha="center", va="bottom",
             fontsize=11, fontweight="bold", color="white")
ax4.set_title("Number of Customers per Cluster",
              fontsize=13, fontweight="bold", color="#58A6FF", pad=12)
ax4.set_xlabel("Cluster")
ax4.set_ylabel("Customer Count")
ax4.grid(True, axis="y")

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("kmeans_results.png", dpi=150, bbox_inches="tight",
            facecolor="#0D1117")
plt.close()
print("  ✔  Plot saved as  →  kmeans_results.png")


# ─────────────────────────────────────────────
# STEP 10: Save Results to CSV
# ─────────────────────────────────────────────
print("\n[STEP 9] Saving Clustered Data...")

output = df.copy()
output["Cluster_Label"] = output["Cluster"].map(lambda c: label_cluster(c))
output.to_csv("Mall_Customers_Clustered.csv", index=False)
print("  ✔  Results saved as  →  Mall_Customers_Clustered.csv")


# ─────────────────────────────────────────────
# STEP 11: Final Summary
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("   FINAL SUMMARY")
print("=" * 60)
print(f"  Algorithm       : K-Means Clustering (k-means++ init)")
print(f"  Optimal K       : {K}")
print(f"  Total Customers : {len(df)}")
print(f"  WCSS (Inertia)  : {kmeans.inertia_:.4f}")
print(f"  Silhouette Score: {silhouette_score(X_scaled, df['Cluster']):.4f}")
print(f"\n  Cluster Breakdown:")
for c in range(K):
    cnt = cluster_counts[c]
    pct = (cnt / len(df)) * 100
    print(f"    Cluster {c} ({cnt} customers, {pct:.1f}%) → {label_cluster(c)}")

print("\n  Output Files:")
print("    ✔  kmeans_results.png          (4-panel visualization)")
print("    ✔  Mall_Customers_Clustered.csv (data with cluster labels)")
print("\n" + "=" * 60)
print("   Task 02 Completed Successfully!")
print("=" * 60)

   TASK 02 — Customer Segmentation: K-Means Clustering

[STEP 1] Loading Dataset...
  ✔  Shape       : 200 rows × 5 columns
  ✔  Columns     : ['CustomerID', 'Gender', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']

  First 5 rows:
 CustomerID Gender  Age  Annual Income (k$)  Spending Score (1-100)
          1   Male   19                  15                      39
          2   Male   21                  15                      81
          3 Female   20                  16                       6
          4 Female   23                  16                      77
          5 Female   31                  17                      40

  Dataset Info:
  Column                         Non-Null     Dtype
  --------------------------------------------------
  CustomerID                     200          int64
  Gender                         200          str
  Age                            200          int64
  Annual Income (k$)             200          int64
  Spending Score (1-100)

# Step 2: Load Dataset

 Dataset Successfully Loaded!

 Dataset Shape: (200, 5)
 Total Customers: 200
 Total Features: 5


# Step 3: Exploratory Data Analysis (EDA)


# Step 4: Data Visualization — EDA Plots


# Step 5: Boxplots — Outlier Detection


In [23]:
# ============================================================
#   STEP 5: BOXPLOT — OUTLIER DETECTION
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle('Outlier Detection — Boxplots', 
             fontsize=16, fontweight='bold')

numeric_cols = ['Age', 'Annual_Income', 'Spending_Score']
colors_box   = ['#4ECDC4', '#45B7D1', '#FF6B6B']
labels_box   = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']

for i, (col, color, label) in enumerate(zip(numeric_cols, colors_box, labels_box)):
    axes[i].boxplot(df[col], patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='red', linewidth=2))
    axes[i].set_title(label, fontsize=13, fontweight='bold')
    axes[i].set_ylabel('Value')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('boxplots.png', dpi=300, bbox_inches='tight')
plt.show()



ModuleNotFoundError: No module named 'matplotlib_inline'

# Step 6: Correlation Heatmap


In [24]:
# ============================================================
#   STEP 6: CORRELATION HEATMAP
# ============================================================

# Convert gender into numeric 
df_corr = df.copy()
df_corr['Gender_Encoded'] = df_corr['Gender'].map({'Male': 0, 'Female': 1})

plt.figure(figsize=(8, 6))

corr_cols = ['Age', 'Annual_Income', 'Spending_Score', 'Gender_Encoded']
corr_matrix = df_corr[corr_cols].corr()

sns.heatmap(corr_matrix,
            annot=True,
            fmt='.2f',
            cmap='coolwarm',
            center=0,
            square=True,
            linewidths=2,
            cbar_kws={"shrink": 0.8},
            xticklabels=['Age', 'Annual\nIncome', 
                         'Spending\nScore', 'Gender'],
            yticklabels=['Age', 'Annual\nIncome', 
                         'Spending\nScore', 'Gender'])

plt.title('Feature Correlation Heatmap', 
          fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()


ModuleNotFoundError: No module named 'matplotlib_inline'

# Step 7: Feature Selection & Data Preprocessing


In [ ]:
# ============================================================
#   STEP 7: FEATURE SELECTION & DATA PREPROCESSING
# ============================================================

# Select 2 main features for clustering 
# (Annual Income & Spending Score — most impactful)
X = df[['Annual_Income', 'Spending_Score']].copy()

print("=" * 55)
print("     SELECTED FEATURES FOR CLUSTERING")
print("=" * 55)
print("     Annual_Income  (k$)")
print("     Spending_Score (1-100)")
print(f"\nFeature Matrix Shape: {X.shape}")

# StandardScaler se data scale karo
scaler  = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("\nData Successfully Scaled with StandardScaler!")
print("\nScaled Data Sample (First 5 rows):")
scaled_df = pd.DataFrame(X_scaled, 
                          columns=['Annual_Income_Scaled', 
                                   'Spending_Score_Scaled'])
print(scaled_df.head().round(4).to_string(index=False))


# Step 8: Optimal K — Elbow Method


In [ ]:
# ============================================================
#   STEP 8: ELBOW METHOD — FIND OPTIMAL K 
# ============================================================

inertia_values = []
k_range        = range(1, 11)

print("Computing Inertia for K = 1 to 10 ...")
print("-" * 40)

for k in k_range:
    km = KMeans(n_clusters=k,
                init='k-means++',
                n_init=10,
                random_state=42)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)
    print(f"   K = {k:2d}  →  Inertia: {km.inertia_:10.2f}")

# Elbow Curve Plot
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_values,
         'o-', linewidth=2.5, markersize=9,
         color='#FF6B6B', markerfacecolor='white',
         markeredgewidth=2)

plt.fill_between(k_range, inertia_values, alpha=0.08, color='#FF6B6B')

# Mark optimal K = 5 
plt.axvline(x=5, color='green', linestyle='--',
            linewidth=2, label='Optimal K = 5')
plt.scatter([5], [inertia_values[4]], 
            color='green', s=200, zorder=5)

plt.xlabel('Number of Clusters (K)', fontsize=13)
plt.ylabel('Inertia (WCSS)', fontsize=13)
plt.title('Elbow Method — Finding Optimal K', 
          fontsize=16, fontweight='bold')
plt.xticks(k_range)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_method.png', dpi=300, bbox_inches='tight')
plt.show()
print("Optimal K Selected = 5")

# Step 9: Silhouette Score Validation


In [ ]:
# ============================================================
#   STEP 9: SILHOUETTE SCORE — VALIDATE K 
# ============================================================

silhouette_scores = []
k_range_sil       = range(2, 11)

print("Computing Silhouette Scores ...")
print("-" * 45)

for k in k_range_sil:
    km     = KMeans(n_clusters=k, init='k-means++',
                    n_init=10, random_state=42)
    labels = km.fit_predict(X_scaled)
    score  = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"   K = {k:2d}  →  Silhouette Score: {score:.4f}")

best_k     = k_range_sil[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)

print(f"\nBest K = {best_k}  |  Best Score = {best_score:.4f}")

# Silhouette Plot
plt.figure(figsize=(10, 6))
bars = plt.bar(k_range_sil, silhouette_scores,
               color=['#2ECC71' if k == best_k else '#AED6F1' 
                      for k in k_range_sil],
               edgecolor='black', linewidth=0.7)

plt.plot(k_range_sil, silhouette_scores,
         'o-', color='#E74C3C', linewidth=2, markersize=7)

plt.xlabel('Number of Clusters (K)', fontsize=13)
plt.ylabel('Silhouette Score', fontsize=13)
plt.title('Silhouette Score Analysis', fontsize=16, fontweight='bold')
plt.xticks(k_range_sil)
plt.axvline(x=best_k, color='darkgreen', linestyle='--',
            linewidth=2, label=f'Best K = {best_k}')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('silhouette_scores.png', dpi=300, bbox_inches='tight')
plt.show()

# Step 10: Finally Train K-Means Model  


In [ ]:
# ============================================================
#   STEP 10: FINAL K-MEANS MODEL TRAINING (K = 5)
# ============================================================

OPTIMAL_K = 5

print(f" Training Final K-Means Model  |  K = {OPTIMAL_K}")
print("=" * 55)

final_kmeans = KMeans(
    n_clusters  = OPTIMAL_K,
    init        = 'k-means++',   # Smart centroid initialization
    n_init      = 10,            # run 10 times for best result 
    max_iter    = 300,           # Maximum iterations
    random_state= 42,            # Reproducibility
    algorithm   = 'lloyd'        # Standard K-Means
)

cluster_labels = final_kmeans.fit_predict(X_scaled)

# Add labels into dataset   
df['Cluster'] = cluster_labels

# Assign cluster names (after analysis)
cluster_name_map = {
    0: 'Careful Spenders',
    1: 'Standard Customers',
    2: 'Target Customers',      # High Income, High Spending
    3: 'Careless Spenders',     # Low Income, High Spending
    4: 'Sensible Customers'     # High Income, Low Spending
}
df['Cluster_Name'] = df['Cluster'].map(cluster_name_map)

print(" Model Successfully Trained!")
print(f"\n Model Info:")
print(f"   • Inertia (WCSS):    {final_kmeans.inertia_:.2f}")
print(f"   • Iterations Taken:  {final_kmeans.n_iter_}")

print(f"\n Cluster Distribution:")
print("-" * 40)
for cid, cname in cluster_name_map.items():
    count = (df['Cluster'] == cid).sum()
    pct   = count / len(df) * 100
    print(f"   Cluster {cid} | {cname:<22} | {count:3d} customers ({pct:.1f}%)")

# Step 11: Cluster Visualization


In [ ]:
# ============================================================
#   STEP 11: CLUSTER VISUALIZATION
# ============================================================

cluster_colors = ['#FF6B6B', '#4ECDC4', '#2ECC71', '#F39C12', '#9B59B6']

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('K-Means Customer Segmentation — Mall Customers Dataset',
             fontsize=16, fontweight='bold')

# --- Plot 1: Annual Income vs Spending Score (Main Plot) ---
for cid in range(OPTIMAL_K):
    mask = df['Cluster'] == cid
    axes[0].scatter(
        df[mask]['Annual_Income'],
        df[mask]['Spending_Score'],
        c=cluster_colors[cid],
        label=cluster_name_map[cid],
        s=100, alpha=0.8,
        edgecolors='black', linewidth=0.5
    )

# plot centroids into original scale 
centroids_orig = scaler.inverse_transform(final_kmeans.cluster_centers_)
axes[0].scatter(
    centroids_orig[:, 0],
    centroids_orig[:, 1],
    c='black', marker='*',
    s=400, zorder=10,
    label='Centroids', edgecolors='white', linewidth=1
)

axes[0].set_xlabel('Annual Income (k$)', fontsize=12)
axes[0].set_ylabel('Spending Score (1-100)', fontsize=12)
axes[0].set_title('Income vs Spending Score — Clusters',
                   fontsize=14, fontweight='bold')
axes[0].legend(fontsize=9, loc='upper left')
axes[0].grid(True, alpha=0.3)

# --- Plot 2: Age vs Spending Score ---
for cid in range(OPTIMAL_K):
    mask = df['Cluster'] == cid
    axes[1].scatter(
        df[mask]['Age'],
        df[mask]['Spending_Score'],
        c=cluster_colors[cid],
        label=cluster_name_map[cid],
        s=100, alpha=0.8,
        edgecolors='black', linewidth=0.5
    )

axes[1].set_xlabel('Age', fontsize=12)
axes[1].set_ylabel('Spending Score (1-100)', fontsize=12)
axes[1].set_title('Age vs Spending Score — Clusters',
                   fontsize=14, fontweight='bold')
axes[1].legend(fontsize=9, loc='upper right')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cluster_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# Step 12: Cluster-wise Detailed Analysis


In [ ]:
# ============================================================
#   STEP 12: CLUSTER-WISE DETAILED ANALYSIS
# ============================================================

print("=" * 65)
print("     DETAILED CLUSTER ANALYSIS — MALL CUSTOMERS")
print("=" * 65)

for cid in range(OPTIMAL_K):
    cdata = df[df['Cluster'] == cid]
    cname = cluster_name_map[cid]
    
    print(f"\n{'='*60}")
    print(f"    CLUSTER {cid}: {cname.upper()}")
    print(f"{'='*60}")
    print(f"    Total Customers : {len(cdata)}")
    
    # Gender breakdown
    male_c   = (cdata['Gender'] == 'Male').sum()
    female_c = (cdata['Gender'] == 'Female').sum()
    print(f"    Gender Split    : Male={male_c} | Female={female_c}")
    
    print(f"\n    Key Statistics:")
    print(f"     • Avg Age             : {cdata['Age'].mean():.1f} yrs")
    print(f"     • Age Range           : {cdata['Age'].min()} – {cdata['Age'].max()} yrs")
    print(f"     • Avg Annual Income   : ${cdata['Annual_Income'].mean():.1f}k")
    print(f"     • Avg Spending Score  : {cdata['Spending_Score'].mean():.1f}/100")
    print(f"     • Income Range        : ${cdata['Annual_Income'].min()}k – ${cdata['Annual_Income'].max()}k")
    print(f"     • Spending Range      : {cdata['Spending_Score'].min()} – {cdata['Spending_Score'].max()}")


# Step 13: Business Insights & Recommendations


In [ ]:
# ============================================================
#  STEP 13: BUSINESS INSIGHTS & RECOMMENDATIONS
# ============================================================

print("=" * 65)
print("     BUSINESS INSIGHTS & MARKETING RECOMMENDATIONS")
print("=" * 65)

insights = {
    'Careful Spenders': {
        'profile'    : "Medium Income, Medium Spending",
        'strategy'   : [
            "  Offer a loyalty rewards program",
            "  Provide EMI options and easy payment plans",
            "  Send seasonal sale notifications"
        ]
    },
    'Standard Customers': {
        'profile'    : "Average Income, Average Spending",
        'strategy'   : [
            "  Analyze purchase history to provide personalized offers",
            "  Offer a subscription-based membership",
            "  Engage them through referral programs"
        ]
    },
    'Target Customers': {
        'profile'    : "High Income, High Spending MOST VALUABLE",
        'strategy'   : [
            "  Activate a VIP / Premium membership program",
            "  Send exclusive luxury product recommendations",
            "  Give early access to new arrivals"
        ]
    },
    'Careless Spenders': {
        'profile'    : "Low Income, High Spending",
        'strategy'   : [
            "  Offer budget-friendly product bundles",
            "  Send flash sale alerts and discount coupons",
            "  Promote app-exclusive deals"
        ]
    },
    'Sensible Customers': {
        'profile'    : "High Income, Low Spending",
        'strategy'   : [
            "  Showcase high-quality premium products",
            "  Run value-for-money campaigns",
            "  Invite them to exclusive member-only events"
        ]
    }
}

for segment, info in insights.items():
    print(f"\n    {segment}")
    print(f"     Profile  : {info['profile']}")
    print(f"     Strategy :")
    for s in info['strategy']:
        print(f"       {s}")

# Step 14: Model Evaluation


In [ ]:
# ============================================================
#   STEP 14: FINAL MODEL EVALUATION
# ============================================================

final_silhouette = silhouette_score(X_scaled, cluster_labels)

print("=" * 55)
print("     FINAL MODEL EVALUATION METRICS")
print("=" * 55)
print(f"\n    Algorithm         : K-Means Clustering")
print(f"    Initialization    : K-Means++")
print(f"    Optimal K         : {OPTIMAL_K}")
print(f"    Inertia (WCSS)    : {final_kmeans.inertia_:.2f}")
print(f"    Silhouette Score  : {final_silhouette:.4f}")
print(f"    Iterations        : {final_kmeans.n_iter_}")
print(f"    Total Customers   : {len(df)}")

print(f"\n    Silhouette Score Interpretation:")
if final_silhouette >= 0.70:
    rating = "  Excellent Clustering!"
elif final_silhouette >= 0.50:
    rating = "  Good Clustering!"
elif final_silhouette >= 0.25:
    rating = "  Fair Clustering"
else:
    rating = "  Poor Clustering — Revisit K"

print(f"     {rating}")
print(f"     Score = {final_silhouette:.4f}  (Range: -1 to +1, Higher = Better)")
